# MLP Baseline — MPP (основная модель, Colab GPU)

Полноразмерная MLP baseline модель для сравнения с Transformer на задаче MPP.

**Запуск**: Google Colab с GPU (Runtime → Change runtime type → GPU).

**Параметры**:
- embed_size=128, num_layers=1, forward_expansion=5
- Количество параметров подобрано для соответствия основной Transformer модели
- 2000 эпох, lr=1e-4, batch_size=256 (pre-collated), mask=25%

In [ ]:
# --- Colab: клонирование репо и установка зависимостей ---
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

if IN_COLAB:
    REPO_URL = "https://github.com/GibaDuliya/ML_project-football-.git"
    BRANCH = "baselines"
    REPO_DIR = Path("/content/ML_project-football-")
    if not REPO_DIR.exists():
        import subprocess
        subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    REQ = REPO_DIR / "requirements.txt"
    if REQ.exists():
        import subprocess
        subprocess.run(["pip", "install", "-q", "-r", str(REQ)], check=True)
    ROOT = REPO_DIR
else:
    ROOT = Path(".").resolve()
    while ROOT.name and not (ROOT / "models").exists():
        ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

In [ ]:
import torch
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Данные и словари

Для Colab: загрузи `data_with_dates.csv` в `/content/` или добавь как датасет.

In [ ]:
from data.preprocessing import preprocess_raw_csv, build_vocab_mappings

# Поиск CSV файла
possible_paths = [
    ROOT / "dataset" / "data_with_dates.csv",
    Path("/content/data_with_dates.csv"),
    Path("/content/drive/MyDrive/data_with_dates.csv"),
]
raw_path = None
for p in possible_paths:
    if p.exists():
        raw_path = p
        break
if raw_path is None:
    raise FileNotFoundError(
        "Не найден data_with_dates.csv. Укажи путь вручную: raw_path = Path('...')"
    )
print("Data CSV:", raw_path)

output_dir_data = str(ROOT / "notebooks" / "baselines" / "MLP_baseline" / "processed")
Path(output_dir_data).mkdir(parents=True, exist_ok=True)

df = preprocess_raw_csv(str(raw_path), output_dir_data)
vocab = build_vocab_mappings(df, output_dir_data)

print(f"Матчей: {df['match_id'].nunique()}")
print(f"players_vocab_size: {vocab['players_vocab_size']}")
print(f"player_pad_token_id: {vocab['player_pad_token_id']}")
print(f"player_mask_token_id: {vocab['player_mask_token_id']}")

## 2. Датасет с pre-collation (как в kaggle ноутбуке)

In [ ]:
from data.dataset import MatchDatasetMPP, PreCollatedDataset
from data.collator import DataCollatorMPP, DataCollatorPreCollated
from torch.utils.data import DataLoader

max_seq_length = 36
sample_batch_size = 256
repeat = 20
dev_ratio = 0.05
seed = 42

ds_full = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

def _collate_filter_none(batch):
    batch = [b for b in batch if b is not None]
    return collator(batch) if batch else None

dataloader_build = DataLoader(
    ds_full, batch_size=sample_batch_size, shuffle=True,
    collate_fn=_collate_filter_none, drop_last=True,
)

print(f"Собираем батчи (repeat={repeat})...")
all_batches = []
for r in range(repeat):
    for batch in dataloader_build:
        if batch is not None:
            all_batches.append(batch)

np.random.seed(seed)
n_batches = len(all_batches)
dev_size = max(1, int(n_batches * dev_ratio))
dev_idx = set(np.random.choice(n_batches, size=dev_size, replace=False).tolist())
train_batches = [all_batches[i] for i in range(n_batches) if i not in dev_idx]
dev_batches = [all_batches[i] for i in range(n_batches) if i in dev_idx]

train_dataset = PreCollatedDataset(train_batches)
eval_dataset = PreCollatedDataset(dev_batches)
collator_for_trainer = DataCollatorPreCollated()

print(f"Батчей: {n_batches}, train: {len(train_batches)}, eval: {len(dev_batches)}")

## 3. Модель

Подбираем `forward_expansion` так, чтобы количество параметров MLP baseline
было максимально близко к основной Transformer модели.

In [ ]:
from baselines.MLP_baseline.MLP_baseline import MLPMaskedPlayerModel
from models.pretrain import MaskedPlayerModel

# Основная модель (Transformer) — для подсчёта параметров
model_transformer = MaskedPlayerModel(
    embed_size=128,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
n_transformer = sum(p.numel() for p in model_transformer.parameters())
print(f"Transformer параметров: {n_transformer:,}")
del model_transformer

# MLP baseline — forward_expansion=5 для приближения к тому же числу параметров
model = MLPMaskedPlayerModel(
    embed_size=128,
    num_layers=1,
    forward_expansion=5,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

n_mlp = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MLP Baseline параметров: {n_mlp:,} (trainable: {n_trainable:,})")
print(f"Разница с Transformer: {abs(n_mlp - n_transformer):,} ({abs(n_mlp - n_transformer) / n_transformer * 100:.1f}%)")

## 4. Обучение

In [ ]:
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp

output_dir = str(ROOT / "notebooks" / "baselines" / "MLP_baseline" / "output_main")
Path(output_dir).mkdir(parents=True, exist_ok=True)

training_config = {
    "output_dir": output_dir,
    "num_train_epochs": 2000,
    "per_device_train_batch_size": 1,   # pre-collated: каждый "сэмпл" уже батч
    "per_device_eval_batch_size": 1,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 1000,
    "eval_strategy": "steps",
    "eval_steps": 1000,
    "save_strategy": "steps",
    "save_steps": 10000,
    "save_total_limit": 3,
    "report_to": "none",
    "seed": seed,
    "load_best_model_at_end": True,
    "metric_for_best_model": "accuracy_top1",
    "greater_is_better": True,
}

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator_for_trainer,
)

print(f"Trainer готов. Параметров: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
best_model_dir = Path(output_dir) / "best_model"

try:
    trainer.train()
finally:
    trainer.save_model(str(best_model_dir))
    print(f"Модель сохранена: {best_model_dir}")

if trainer.state.log_history:
    pd.DataFrame(trainer.state.log_history).to_csv(
        Path(output_dir) / "metrics.csv", index=False
    )
    print("Метрики сохранены в metrics.csv")

## 5. Оценка на валидации

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

eval_results = trainer.evaluate()
print("\nРезультаты на валидации:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## 6. Визуализация метрик

In [ ]:
import matplotlib.pyplot as plt

metrics_df = pd.DataFrame(trainer.state.log_history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
train_loss = metrics_df.dropna(subset=["loss"])
eval_loss = metrics_df.dropna(subset=["eval_loss"])
axes[0].plot(train_loss["step"], train_loss["loss"], label="train")
axes[0].plot(eval_loss["step"], eval_loss["eval_loss"], label="eval")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss")
axes[0].legend()

# Top-1 accuracy
eval_acc = metrics_df.dropna(subset=["eval_accuracy_top1"])
axes[1].plot(eval_acc["step"], eval_acc["eval_accuracy_top1"])
axes[1].set_xlabel("step")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Eval Top-1 Accuracy")

# Top-3 accuracy
axes[2].plot(eval_acc["step"], eval_acc["eval_accuracy_top3"])
axes[2].set_xlabel("step")
axes[2].set_ylabel("accuracy")
axes[2].set_title("Eval Top-3 Accuracy")

plt.tight_layout()
plt.savefig(Path(output_dir) / "metrics_plot.png", dpi=150)
plt.show()
print("Done!")